In [1]:
from openai import OpenAI
import openai
import os
import supabase
from time import sleep
from supabase import create_client, Client

In [2]:
openai.api_key = os.environ.get("OPENAI_API_KEY")

# Initialize supabase client
url: str = os.environ.get("SUPABASE_URL")
key: str = os.environ.get("SUPABASE_KEY")

In [4]:
client = OpenAI()
supabase_client: Client = create_client(url, key)

### How to get an embedding from OpenAI

In [5]:
!cat docs/doc1.txt

cat says meow

In [6]:
!cat docs/doc2.txt

dog will bark

In [9]:
with open("docs/doc1.txt", "r") as f:
    text = f.read()
    response = client.embeddings.create(input=text, model="text-embedding-3-small", encoding_format="float")
    embedding_doc1 = response.data[0].embedding
    print(embedding_doc1)

[-0.01820233, -0.050404314, 0.009908303, 0.02457593, -0.0049819835, 0.016128823, -0.023838373, 0.028639447, -0.038575582, -0.0005040257, 0.025174323, -0.015697423, 0.009275117, 0.012197509, 0.027122585, 0.009650853, -0.009017669, 0.0019778335, -0.014667627, 0.044782188, -0.0019673964, 0.00029941477, -0.0077652144, -0.001476852, -0.012274048, 0.02276683, -0.007010263, 0.023768792, 0.021695286, -0.025522228, -0.02790189, -0.042611267, -0.061509404, -0.092681594, -0.056861408, 0.022878159, -0.008516687, -0.032925624, 0.021667453, 0.0055490667, -0.034846053, -0.017144702, 0.035569694, -0.016434979, 0.0010210979, -0.0607301, -0.025035163, 0.030504212, 0.042694762, -0.00018721577, -0.011362541, 0.0087532615, 0.019023383, 0.05062697, -0.004153972, -0.0061544194, 0.046730448, 0.021486543, -0.010040506, -0.015377351, 0.006857185, -0.003948709, 0.011272086, -0.038770407, 0.024896001, 0.03072687, 0.021973608, -0.026732933, -0.025188241, -0.012134887, -0.007973957, 0.053410202, -0.054495662, 0.041

In [10]:
with open("docs/doc2.txt", "r") as f:
    text = f.read()
    response = client.embeddings.create(input=text, model="text-embedding-3-small", encoding_format="float")
    embedding_doc2 = response.data[0].embedding
    print(embedding_doc2)

[-0.027630325, 0.0033874295, -0.0070945984, 0.0024719487, -0.031925693, -0.011914823, -0.0057734107, 0.007655651, -0.02883689, 0.0058307224, 0.0002626163, 0.013778965, 0.009073365, -0.007957292, -0.015588812, 0.0060418714, -0.028981676, 0.04121624, -0.057625514, -0.004467305, -0.028040556, 0.08383209, -0.020234086, -0.015781863, 0.001211089, 0.03240832, 0.014032343, 0.010702226, 0.02221285, 0.013272208, -0.021114876, -0.021983603, 0.010310093, -0.053137098, 0.0365589, 0.0085545415, -0.034000985, -0.030429553, 0.051061805, 0.036486506, -0.042808905, -0.063079186, 0.02625484, 3.5584224e-05, 0.01941362, -0.0039002195, 0.0013309914, 0.00947153, 0.014394313, 0.022828199, 0.016614392, 0.016445473, 0.02548264, 0.066409305, 0.025265459, -0.023033313, 0.004735765, -0.0069256797, 0.022357639, -0.009031135, -0.016735049, -0.045584004, 0.06703672, -0.03091218, -0.00064400374, -0.003088805, 0.028137082, -0.032167006, 0.050820492, -0.0125844665, 0.010080845, 0.00074015185, -0.050723966, 0.06385139, 

### Insert vector embeddings to database

- In the vector database, we have defined a TABLE with doc_name and embedding vector
- We also have a function that returns the similarity of a query with all of the documents stored in it

In [11]:
query = "dog" # Define your query here
embedding_query = client.embeddings.create(input=query, model="text-embedding-3-small", encoding_format="float").data[0].embedding
response = supabase_client.rpc("query_documents", {"query_embedding": embedding_query}).execute()

In [13]:
response.data

[{'doc_name': 'docs/doc1.txt', 'similarity': 0.359976291656497},
 {'doc_name': 'docs/doc2.txt', 'similarity': 0.545763698472757}]